In [ ]:
import requests
import time
import pandas as pd
import random
from tqdm import tqdm

In [22]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
    'Referer': 'https://tiki.vn/lam-dep-suc-khoe/c1520',
    'Accept': 'application/json, text/plain, */*',
    'Connection': 'keep-alive',
}

params = {
    'limit': '10',
    'sort': 'top_seller',
    'category': '1520',
    'page': '1',
    'urlKey': 'lam-dep-suc-khoe',
}

product_id = []
for i in range(1, 6):
    params['page'] = str(i)
    response = requests.get('https://tiki.vn/api/personalish/v1/blocks/listings', headers=headers, params=params)

    if response.status_code == 200:
        print(f'Page {i}: Request success!!!')
        json_data = response.json()
        
        if 'data' in json_data:
            for record in json_data.get('data', []):
                product_id.append({'id': record.get('id')})
        else:
            print(f'Page {i}: No data found!')
    else:
        print(f'Page {i}: Request failed with status {response.status_code}')
    
    time.sleep(random.uniform(3, 10))  # Random delay để tránh bị chặn

df = pd.DataFrame(product_id)
df.to_csv('TestID_LamDepSK.csv', index=False)
print("Done! Data saved to TestID_LamDepSK.csv1")

Page 1: Request success!!!
Page 2: Request success!!!
Page 3: Request success!!!
Page 4: Request success!!!
Page 5: Request success!!!
Done! Data saved to TestID_LamDepSK.csv1


In [ ]:
# # Hàm parser dữ liệu sản phẩm
# def parser_product(json):
#     return {
#         'id': json.get('id'),
#         'name': json.get('name'),
#         #'breadcrumbs' : json.get('breadcrumbs'),
        
#         'url_key': json.get('url_key'),
#         'url_product': json.get('breadcrumbs', [{}])[-1].get('url'),
#         'image_base_url': json.get('images', [{}])[0].get('base_url'),
#         'rating_average': json.get('rating_average'),
#         'review_count': json.get('review_count'),
#         'quantity_sold': json.get('quantity_sold'),
#         'brand_name': json.get('brand', {}).get('name'), 
#         'category': [bc.get('name') for bc in json.get('breadcrumbs', [])[:-1]],
#         'current_seller': json.get('current_seller', {}).get('name'), 
#         'inventory_status': json.get('inventory_status'),
#         'stock_item_qty': json.get('stock_item', {}).get('qty'),
#         'original_price' : json.get('original_price'),
#         'discount': json.get('discount'),
#         'URL_seller': json.get('current_seller', {}).get('link'),
#         'Rating_seller': json.get('current_seller', {}).get('rating'),
#         'Seller_Type': json.get('current_seller', {}).get('name'),
#         'Seller_Location' : json.get('current_seller', {}).get('type')
        
        
#         # 'review_count': json.get('review_count'),
#         # 'order_count': json.get('order_count'),
#         # 'is_visible': json.get('is_visible'),
#         # 'stock_item_qty': json.get('stock_item', {}).get('qty'),
#         # 'stock_item_max_sale_qty': json.get('stock_item', {}).get('max_sale_qty'),
#         # 'product_name': json.get('meta_title'),
#         # 'brand_id': json.get('brand', {}).get('id'),
#         # 'brand_name': json.get('brand', {}).get('name')
#     }

# # Đọc danh sách ID sản phẩm
# df_id = pd.read_csv('TestID_LamDepSK.csv')
# p_ids = df_id.id.to_list()
# result = []

# # Bắt đầu thu thập dữ liệu
# for pid in tqdm(p_ids, total=len(p_ids)):
#     retries = 3  # Số lần thử lại nếu request thất bại
#     while retries > 0:
#         response = requests.get(f'https://tiki.vn/api/v2/products/{pid}', headers=headers)

#         if response.status_code == 200:
#             print(f'Crawl data {pid} success!')
#             result.append(parser_product(response.json()))
#             break  # Nếu thành công, thoát khỏi vòng lặp retry
#         elif response.status_code == 429:
#             print(f'Too many requests! Waiting before retrying {pid}...')
#             time.sleep(random.uniform(5, 10))  # Chờ ngẫu nhiên để tránh bị chặn
#         elif response.status_code == 404:
#             print(f'Product {pid} not found (404). Skipping...')
#             break  # Không thử lại nếu sản phẩm không tồn tại
#         else:
#             print(f'Failed to fetch data for {pid}, status code: {response.status_code}')
#         retries -= 1
    
#     # Chờ ngẫu nhiên giữa các request
#     time.sleep(random.uniform(1, 5)) 

# # Lưu dữ liệu ra file CSV
# df_product = pd.DataFrame(result)
# df_product.to_csv('LamDepVaSucKhoe1.csv', index=False)
# print("Done! Data saved to LamDepVaSucKhoe1.csv")


In [ ]:
import requests
import pandas as pd
import time
import random
from tqdm import tqdm

# headers = {
#     "User-Agent": "Mozilla/5.0"
# }

def get_seller_info(seller_id):
    """Lấy thông tin người bán từ API"""
    url = f'https://api.tiki.vn/product-detail/v2/widgets/seller?seller_id={seller_id}'
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        seller_data = response.json().get('data', {}).get('seller', {})
        return {
            'seller_rating': seller_data.get('avg_rating_point'),
            'review_count': seller_data.get('review_count'),
            'url_seller': seller_data.get('url')
        }
    return {'seller_rating': None, 'review_count': None, 'url_seller': None}


def parser_product(json):
    """Trích xuất thông tin sản phẩm từ API"""
    seller_id = json.get('current_seller', {}).get('id')
    seller_info = get_seller_info(seller_id) if seller_id else {'Seller Rating': None, 'review_count': None}
    
    return {
        'id': json.get('id'),
        'name': json.get('name'),
        'url_key': json.get('url_key'),
        'url_product': json.get('breadcrumbs', [{}])[-1].get('url'),
        'image_base_url': json.get('images', [{}])[0].get('base_url'),
        'rating_average': json.get('rating_average'),
        'review_count': json.get('review_count'),
        'quantity_sold': json.get('quantity_sold'),
        'brand_name': json.get('brand', {}).get('name'), 
        'category': [bc.get('name') for bc in json.get('breadcrumbs', [])[:-1]],
        'category_id': [bc.get('category_id') for bc in json.get('breadcrumbs', [])[:-1]],
        'current_seller': json.get('current_seller', {}).get('name'), 
        'inventory_status': json.get('inventory_status'),
        'stock_item_qty': json.get('stock_item', {}).get('qty'),
        'original_price': json.get('original_price'),
        'discount': json.get('discount'),
        'price': json.get('price'),
        
        # 'url_seller': json.get('current_seller', {}).get('link'),
        # 'Seller_Type': json.get('current_seller', {}).get('name'),
        # 'Seller_Location': json.get('current_seller', {}).get('type'),
        # 'seller_rating': seller_info['seller_rating'],
        # 'review_count': seller_info['review_count']
        'seller_rating': seller_info['seller_rating'],
        'review_count_seller': seller_info['review_count'],  # Review của người bán
        'url_seller': seller_info['url_seller'],
        'Seller_Type': json.get('current_seller', {}).get('name')
    }

# Đọc danh sách ID sản phẩm
df_id = pd.read_csv('TestID_LamDepSK.csv')
p_ids = df_id.id.to_list()
result = []

# Bắt đầu thu thập dữ liệu
for pid in tqdm(p_ids, total=len(p_ids)):
    retries = 3
    while retries > 0:
        response = requests.get(f'https://tiki.vn/api/v2/products/{pid}', headers=headers)
        
        if response.status_code == 200:
            print(f'Crawl data {pid} success!')
            result.append(parser_product(response.json()))
            break
        elif response.status_code == 429:
            print(f'Too many requests! Waiting before retrying {pid}...')
            time.sleep(random.uniform(5, 10))
        elif response.status_code == 404:
            print(f'Product {pid} not found (404). Skipping...')
            break
        else:
            print(f'Failed to fetch data for {pid}, status code: {response.status_code}')
        retries -= 1
    
    time.sleep(random.uniform(1, 5))

# Lưu dữ liệu ra file CSV
df_product = pd.DataFrame(result)
df_product.to_csv('LamDepVaSucKhoe1.csv', index=False)
print("Done! Data saved to LamDepVaSucKhoe1.csv")


In [1]:
from sqlalchemy import create_engine, text

# Thông tin MySQL
db_config = {
    "host": "localhost",
    "user": "myuser",
    "password": "mypassword",
    "database": "mydatabase",
}

# Kết nối MySQL
engine = create_engine(f"mysql+pymysql://{db_config['user']}:{db_config['password']}@{db_config['host']}/{db_config['database']}")


# Tạo bảng nếu chưa tồn tại
create_table_query = """
CREATE TABLE IF NOT EXISTS products (
    id INT PRIMARY KEY,
    name VARCHAR(255),
    url_key VARCHAR(255),
    url_product TEXT,
    image_base_url TEXT,
    rating_average FLOAT,
    review_count INT,
    quantity_sold INT,
    brand_name VARCHAR(255),
    category TEXT,
    category_id TEXT,
    current_seller VARCHAR(255),
    inventory_status VARCHAR(50),
    stock_item_qty INT,
    original_price FLOAT,
    discount FLOAT,
    price FLOAT,
    seller_rating FLOAT,
    review_count_seller INT,
    url_seller TEXT,
    seller_type VARCHAR(255)
);
"""
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()

In [43]:
import pandas as pd

In [57]:
df = pd.read_csv(r'C:\Users\leebo\OneDrive\Documents\internTMA\TMA_Internship_DE_Project\crawl_tiki_API\LamDepVaSucKhoe1.csv')

In [58]:
df.head()

,id,name,url_key,url_product,image_base_url,rating_average,review_count,quantity_sold,brand_name,category,...,current_seller,inventory_status,stock_item_qty,original_price,discount,price,seller_rating,review_count_seller,url_seller,Seller_Type
0,276003962,Nước tẩy trang bí đao Cocoon 500ml,nuoc-tay-trang-bi-dao-500ml-p276003962,https://tiki.vn/nuoc-tay-trang-bi-dao-500ml-p2...,https://salt.tikicdn.com/ts/product/9d/64/ac/6...,5.0,11,"{'text': 'Đã bán 7k', 'value': 7126}",The Cocoon Original Vietnam,"['Làm Đẹp - Sức Khỏe', 'Chăm sóc da mặt', 'Dun...",...,Tiki Trading,available,1000,295000,0,295000,4.6740,5532472,https://tiki.vn/cua-hang/tiki-trading,Tiki Trading
1,275970393,Nước dưỡng tóc tinh dầu bưởi Cocoon 140ml - NEW,nuoc-duong-toc-tinh-dau-buoi-140ml-new-p275970393,https://tiki.vn/nuoc-duong-toc-tinh-dau-buoi-1...,https://salt.tikicdn.com/ts/product/98/2b/f6/8...,4.7,17,"{'text': 'Đã bán 9k', 'value': 9554}",The Cocoon Original Vietnam,"['Làm Đẹp - Sức Khỏe', 'Chăm sóc tóc và da đầu...",...,Tiki Trading,available,1000,154688,0,154688,4.6740,5532472,https://tiki.vn/cua-hang/tiki-trading,Tiki Trading
2,273598230,Máy Tăm Nước Không Dây LocknLock Cordless Oral...,may-tam-nuoc-khong-day-locknlock-cordless-oral...,https://tiki.vn/may-tam-nuoc-khong-day-locknlo...,https://salt.tikicdn.com/ts/product/14/30/51/1...,4.7,1979,"{'text': 'Đã bán 11k', 'value': 11116}",LocknLock,"['Làm Đẹp - Sức Khỏe', 'Chăm sóc răng miệng', ...",...,Tiki Trading,available,1000,1263000,253000,1010000,4.6740,5532472,https://tiki.vn/cua-hang/tiki-trading,Tiki Trading
3,191680827,Kem chống nắng nâng tông kiềm dầu innisfree To...,kem-chong-nang-nang-tong-kiem-dau-innisfree-to...,https://tiki.vn/kem-chong-nang-nang-tong-kiem-...,https://salt.tikicdn.com/ts/product/7f/6d/54/8...,4.8,140,"{'text': 'Đã bán 1k', 'value': 1512}",Innisfree,"['Làm Đẹp - Sức Khỏe', 'Chăm sóc da mặt', 'Kem...",...,Tiki Trading,available,1000,450000,95000,355000,4.6740,5532472,https://tiki.vn/cua-hang/tiki-trading,Tiki Trading
4,186434710,Túi chườm nóng thảo dược giảm đau nhức mỏi mắt...,tui-chuom-nong-thao-duoc-giam-dau-nhuc-moi-mat...,https://tiki.vn/tui-chuom-nong-thao-duoc-giam-...,https://salt.tikicdn.com/ts/product/67/24/fc/c...,4.7,283,"{'text': 'Đã bán 4k', 'value': 4999}",Hapaku,"['Làm Đẹp - Sức Khỏe', 'Máy Massage & Thiết bị...",...,Hapaku Officialstore,available,1000,359000,0,359000,4.6472,2092,https://tiki.vn/cua-hang/hapaku-officialstore,Hapaku Officialstore


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   50 non-null     int64  
 1   name                 50 non-null     object 
 2   url_key              50 non-null     object 
 3   url_product          50 non-null     object 
 4   image_base_url       50 non-null     object 
 5   rating_average       50 non-null     float64
 6   review_count         50 non-null     int64  
 7   quantity_sold        50 non-null     object 
 8   brand_name           50 non-null     object 
 9   category             50 non-null     object 
 10  category_id          50 non-null     object 
 11  current_seller       50 non-null     object 
 12  inventory_status     50 non-null     object 
 13  stock_item_qty       50 non-null     int64  
 14  original_price       50 non-null     int64  
 15  discount             50 non-null     int64

In [40]:
df['breadcrumbs'].unique()

array(["['Làm Đẹp - Sức Khỏe', 'Chăm sóc da mặt', 'Dung dịch tẩy trang']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc tóc và da đầu', 'Dưỡng tóc, ủ tóc', 'Serum, dầu dưỡng tóc', 'Dưỡng mọc tóc']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc răng miệng', 'Máy tăm nước', 'Máy tăm nước cầm tay']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc da mặt', 'Kem chống nắng']",
       "['Làm Đẹp - Sức Khỏe', 'Máy Massage & Thiết bị chăm sóc sức khỏe', 'Túi chườm', 'Chườm nóng']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc cá nhân', 'Chăm sóc tay, chân', 'Nước rửa tay']",
       "['Làm Đẹp - Sức Khỏe', 'Trang điểm ', 'Dụng cụ trang điểm', 'Gương trang điểm']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc tóc và da đầu', 'Dầu gội, dầu xả', 'Dầu Gội', 'Dầu gội sạch gàu']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc cá nhân', 'Dung dịch vệ sinh', 'Dung dịch vệ sinh nam']",
       "['Làm Đẹp - Sức Khỏe', 'Chăm sóc răng miệng', 'Bàn chải đánh răng', 'Bàn chải điện', 'Thân bàn chải điện', 'Bàn chải điện sóng âm']",
    

# Dùng code bên dưới khi thực hiện crawl với số lượng page và dữ liệu nhiều.

In [ ]:
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # Hàm parser dữ liệu sản phẩm
# def parser_product(json):
#     return {
#         'id': json.get('id'),
#         'name': json.get('name'),
#         'short_description': json.get('short_description'),
#         'price': json.get('price'),
#         'list_price': json.get('list_price'),
#         'quantity_sold': json.get('quantity_sold'),
#         'discount': json.get('discount'),
#         'discount_rate': json.get('discount_rate'),
#         'review_count': json.get('review_count'),
#         'order_count': json.get('order_count'),
#         'inventory_status': json.get('inventory_status'),
#         'is_visible': json.get('is_visible'),
#         'stock_item_qty': json.get('stock_item', {}).get('qty'),
#         'stock_item_max_sale_qty': json.get('stock_item', {}).get('max_sale_qty'),
#         'product_name': json.get('meta_title'),
#         'brand_id': json.get('brand', {}).get('id'),
#         'brand_name': json.get('brand', {}).get('name')
#     }

# # Đọc danh sách ID sản phẩm
# df_id = pd.read_csv('ID_ThoiTrangNam1.csv')
# p_ids = df_id.id.to_list()

# # Danh sách User-Agent để tránh bị chặn
# USER_AGENTS = [
#     'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
#     'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36',
#     'Mozilla/5.0 (Windows NT 10.0; WOW64; rv:92.0) Gecko/20100101 Firefox/92.0'
# ]

# # Hàm gửi request lấy dữ liệu sản phẩm
# def fetch_product(pid):
#     retries = 3
#     while retries > 0:
#         headers = {'User-Agent': random.choice(USER_AGENTS)}  # Random User-Agent
#         response = requests.get(f'https://tiki.vn/api/v2/products/{pid}', headers=headers)
        
#         if response.status_code == 200:
#             if response.text.strip():  # Kiểm tra response không rỗng
#                 try:
#                     return parser_product(response.json())  
#                 except requests.JSONDecodeError:
#                     print(f'⚠️ Lỗi JSONDecodeError cho sản phẩm {pid}, response: {response.text[:200]}')
#             else:
#                 print(f'⚠️ Phản hồi rỗng cho sản phẩm {pid}')
#         elif response.status_code == 404:
#             return None  # Sản phẩm không tồn tại
#         elif response.status_code == 429:
#             wait_time = random.uniform(5, 10)
#             print(f'⚠️ Quá nhiều request (429). Đợi {wait_time:.2f}s trước khi thử lại...')
#             time.sleep(wait_time)

#         retries -= 1
#         time.sleep(random.uniform(2, 5))  # Tránh spam request

#     print(f'❌ Bỏ qua sản phẩm {pid} sau {3 - retries} lần thử')
#     return None

# # Thu thập dữ liệu đa luồng
# result = []
# max_workers = 10  # Tùy chỉnh số luồng
# with ThreadPoolExecutor(max_workers=max_workers) as executor:
#     futures = {executor.submit(fetch_product, pid): pid for pid in p_ids}
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         data = future.result()
#         if data:
#             result.append(data)

# # Lưu dữ liệu ra file CSV
# df_product = pd.DataFrame(result)
# df_product.to_csv('tet.csv', index=False)
# print("✅ Done! Data saved to tet.csv")

  4%|▍         | 21/500 [00:03<01:26,  5.55it/s]


⚠️ Quá nhiều request (429). Đợi 7.25s trước khi thử lại...
⚠️ Quá nhiều request (429). Đợi 8.10s trước khi thử lại...
⚠️ Quá nhiều request (429). Đợi 8.17s trước khi thử lại...
